# CogniVoice Component D - GPU runner
Thin notebook: clone repo, install, extract features, train, save outputs.
All real code lives in the git repo, this notebook only runs it.

Setup on Kaggle: enable GPU (T4) in Settings, add your datasets
(MELD wavs + metadata, EmoTa wavs) as Kaggle Dataset inputs.

In [ ]:
# 1. get the code
!git clone https://github.com/Prathikesh/cognivoice-component-d.git
%cd cognivoice-component-d

In [ ]:
# 2. install dependencies (few minutes on first run)
!pip install -q -r requirements.txt

In [ ]:
# 3. build metadata csvs pointing at the Kaggle dataset inputs
# adjust these two paths to match your Kaggle dataset names
MELD_WAV_DIR = '/kaggle/input/meld-wavs'
EMOTA_DIR = '/kaggle/input/emota'

!python -m src.datasets.meld --meld-root {MELD_WAV_DIR} --wav-dir {MELD_WAV_DIR} --out data/metadata_meld.csv
!python -m src.datasets.emota --emota-root {EMOTA_DIR} --out data/metadata_emota.csv

In [ ]:
# 4. extract and cache features (the slow step, runs the frozen encoder)
!python scripts/extract_features.py --metadata data/metadata_meld.csv --out data/features_meld.npz --encoder plus_large
!python scripts/extract_features.py --metadata data/metadata_emota.csv --out data/features_emota.npz --encoder plus_large

In [ ]:
# 5. train: english first, then multilingual fine-tune
!python scripts/train_fusion.py --features data/features_meld.npz --out models/fusion_v2_en.pt
!python scripts/train_fusion.py --features data/features_meld.npz data/features_emota.npz --out models/fusion_v2_multi.pt

In [ ]:
# 6. copy results to /kaggle/working so they appear in the output tab
!cp models/*.pt models/*.json /kaggle/working/ 2>/dev/null; ls -lh /kaggle/working/